In [90]:
# Install dependencies
%pip install anthropic python-dotenv

Note: you may need to restart the kernel to use updated packages.


In [91]:
# Load env variables
from dotenv import load_dotenv

load_dotenv()

True

In [92]:
# Create an API client
from anthropic import Anthropic

client = Anthropic()
model = "claude-sonnet-4-5"

In [93]:
def add_user_message(messages, text):
    user_message = {"role": "user", "content": text}
    messages.append(user_message)

def add_assistant_message(messages, text):
    assistant_message = {"role": "assistant", "content": text}
    messages.append(assistant_message)

def chat(messages, system=None, temperature=1.0, stop_sequences=[]):
    params = {
        "model": model,
        "max_tokens": 1000,
        "messages": messages,
        "extra_body": {"temperature": temperature},
        "stop_sequences": stop_sequences
    }

    if system:
        params["system"] = system

    message = client.messages.create(**params)
    return message.content[0].text

In [94]:
import json

def generate_dataset():
    prompt = """
Generate an evaluation datGenera un conjunto de datos de evaluación para evaluar *prompts*. Este conjunto de datos se utilizará para evaluar *prompts* que generen código Python, JSON o expresiones regulares (Regex) específicamente para tareas relacionadas con AWS. Genera una matriz de objetos JSON, donde cada uno represente una tarea que requiera Python, JSON o una expresión regular para completarse.

Ejemplo de salida:
```json
[
  {
    "task": "Descripción de la tarea",
    "format": "json" o "python" o "regex"
  },
  ...otros
]
```

* Céntrate en tareas que puedan resolverse escribiendo una única función de Python, un único objeto JSON o una única expresión regular.
* Céntrate en tareas que no requieran escribir mucho código.

Por favor generar 3 objetos.
"""
    messages = []
    add_user_message(messages, prompt)
    add_assistant_message(messages, "```json")
    text = chat(messages, stop_sequences=["```"])
    return json.loads(text)


In [95]:
dataset = generate_dataset()

with open("dataset.json", "w") as f:
    json.dump(dataset, f, indent=2)

In [96]:
def run_prompt(test_case):
    """Combina el prompt y la entrada del caso de prueba, y luego devuelve el resultado."""
    prompt = f"""
Por favor resolver la siguiente tarea:

{test_case["task"]}

* Responder solamente en formato Python, JSON o una Regex plano.
* No agregar código comentado o comentarios de explicación.
"""
    
    messages = []
    add_user_message(messages, prompt)
    add_assistant_message(messages, "```code")
    output = chat(messages, stop_sequences=["```"])
    return output

In [97]:
def grade_by_model(test_case, output):
    eval_prompt = f"""
Eres un experto en la revisión de código de AWS. Tu tarea consiste en evaluar la siguiente solución generada por IA.

Tarea Original:
<task>
{test_case["task"]}
</task>

Solución a evaluar:
<solution>
{output}
</solution>

Formato de Salida
Proporcione su evaluación como un objeto JSON estructurado con los siguientes campos, en este orden específico:
- "strengths": Un array de 1 a 3 fortalezas clave
- "weaknesses": Un array de 1 a 3 áreas clave de mejora
- "reasoning": Una explicación concisa de su evaluación general.
- "score": Un número entre 1-10

Responde en formato JSON. Mantén la respuesta concisa y directa.
Ejemplo de estructura de respuesta:
{{
    "strengths": string[],
    "weaknesses": string[],
    "reasoning": string,
    "score": number
}}
    """

    messages = []
    add_user_message(messages, eval_prompt)
    add_assistant_message(messages, "```json")
    eval_text = chat(messages, stop_sequences=["```"])
    return json.loads(eval_text)

In [98]:
# Functions to validate the output structure
import re
import ast


def validate_json(text):
    try:
        json.loads(text.strip())
        return 10
    except json.JSONDecodeError:
        return 0


def validate_python(text):
    try:
        ast.parse(text.strip())
        return 10
    except SyntaxError:
        return 0


def validate_regex(text):
    try:
        re.compile(text.strip())
        return 10
    except re.error:
        return 0


def grade_syntax(response, test_case):
    format = test_case["format"]
    if format == "json":
        return validate_json(response)
    elif format == "python":
        return validate_python(response)
    else:
        return validate_regex(response)

In [99]:
def run_test_case(test_case):
    """Ejecutar run_prompt luego evalúa el resultado"""
    output = run_prompt(test_case)

    model_grade = grade_by_model(test_case, output)
    model_score = model_grade["score"]
    reasoning = model_grade["reasoning"]

    syntax_score = grade_syntax(output, test_case)

    score = (model_score + syntax_score) / 2
    
    return {
        "output": output,
        "test_case": test_case,
        "score": score,
        "reasoning": reasoning
    }

In [100]:
from statistics import mean

def run_eval(dataset):
    """Carga el conjunto de datos y llama a run_test_case con cada caso."""
    results = []
    
    for test_case in dataset:
        result = run_test_case(test_case)
        results.append(result)

    average_score = mean([result["score"] for result in results])
    print(f"Score Promedio: {average_score}")
    
    return results

In [101]:
with open("dataset.json", "r") as f:
    dataset = json.load(f)

results = run_eval(dataset)

Score Promedio: 8.333333333333334


In [102]:
print(json.dumps(results, indent=2))

[
  {
    "output": "\nimport re\n\ndef extraer_nombres_buckets(arns):\n    nombres_buckets = []\n    patron = r'arn:aws:s3:::([^/]+)'\n    for arn in arns:\n        match = re.search(patron, arn)\n        if match:\n            nombres_buckets.append(match.group(1))\n    return nombres_buckets\n",
    "test_case": {
      "task": "Escribe una funci\u00f3n Python que tome una lista de ARNs de S3 y extraiga \u00fanicamente los nombres de los buckets",
      "format": "python"
    },
    "score": 8.5,
    "reasoning": "La soluci\u00f3n es funcionalmente correcta para el caso de uso b\u00e1sico y utiliza regex apropiadamente para extraer nombres de buckets de ARNs est\u00e1ndar de S3. Sin embargo, carece de validaci\u00f3n de entrada, manejo de errores y robustez para casos edge. Para un entorno de producci\u00f3n, necesitar\u00eda validaci\u00f3n de tipos, manejo de excepciones y posiblemente logging para ARNs inv\u00e1lidos."
  },
  {
    "output": "\n{\n  \"Version\": \"2012-10-17\",\n